### Load df_artists_final

In [ ]:
import pandas as pd

df_artists_final = pd.read_csv(
    'df_artists_final.csv',
    index_col=0
)
df_artists_final = df_artists_final.reset_index()

print(df_artists_final.shape)
print(df_artists_final.columns.tolist())

## Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Separate features and target
X = df_artists_final.drop(columns=['top_20_hitmaker'])
y = df_artists_final['top_20_hitmaker']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print('')
print('y value counts:')
print(y.value_counts())
print('')
print('y class balance:')
print(y.value_counts(normalize=True).round(3))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')

In [ ]:
# AdaBoost does not natively accept NaN values.
# We impute with column medians, fitting only on the training set to prevent data leakage.

imputer = SimpleImputer(strategy='median')
X_train_clean = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_clean = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(f'NaN in train: {X_train_clean.isna().sum().sum()}')
print(f'NaN in test:  {X_test_clean.isna().sum().sum()}')

In [ ]:
# Baseline comparison: Dummy classifier vs AdaBoost with default decision tree weak learners.
# AdaBoost here uses DecisionTreeClassifier(max_depth=1) — classic decision stumps.
# This is the original AdaBoost formulation (Freund & Schapire, 1997), unlike ml_sandbox_19_adaboost_linear
# which used linear (SGDClassifier) weak learners.
# Decision stumps can capture non-linear boundaries through ensemble voting,
# while linear weak learners cannot — this is the key difference to explore.
# 5-fold stratified CV to preserve class balance across folds.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'neg_log_loss']

base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42)

models_baseline = {
    'Dummy':    DummyClassifier(strategy='stratified', random_state=42),
    'AdaBoost': AdaBoostClassifier(estimator=base_estimator, n_estimators=100,
                                   learning_rate=1.0, random_state=42),
}

baseline_results = []
for name, model in models_baseline.items():
    cv = cross_validate(model, X_train_clean, y_train, cv=skf,
                        scoring=scoring, return_train_score=True)
    baseline_results.append({
        'Model':           name,
        'Accuracy':        cv['test_accuracy'].mean(),
        'Precision':       cv['test_precision'].mean(),
        'Recall':          cv['test_recall'].mean(),
        'F1':              cv['test_f1'].mean(),
        'ROC-AUC (CV)':    cv['test_roc_auc'].mean(),
        'ROC-AUC (Train)': cv['train_roc_auc'].mean(),
        'Overfit Gap':     cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean(),
        'Log Loss':        -cv['test_neg_log_loss'].mean(),
    })

df_baseline = pd.DataFrame(baseline_results).set_index('Model').round(3)
print(df_baseline)

In [ ]:
# Performance vs n_estimators: evaluate CV AUC and overfit gap at different
# numbers of estimators. Decision tree AdaBoost is more prone to overfitting
# than linear AdaBoost (ml_sandbox_19_adaboost_linear), so this curve may show a clearer peak.
# We test both max_depth=1 (stumps) and max_depth=2 side-by-side.

n_range = list(range(10, 310, 20))
results_by_depth = {}

for depth in [1, 2]:
    depth_results = []
    for n in n_range:
        fold_train_auc, fold_val_auc = [], []
        for train_idx, val_idx in skf.split(X_train_clean, y_train):
            X_tr, X_val = X_train_clean.iloc[train_idx], X_train_clean.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            base = DecisionTreeClassifier(max_depth=depth, random_state=42)
            ada  = AdaBoostClassifier(estimator=base, n_estimators=n,
                                      learning_rate=1.0, random_state=42)
            ada.fit(X_tr, y_tr)
            fold_val_auc.append(roc_auc_score(y_val, ada.predict_proba(X_val)[:, 1]))
            fold_train_auc.append(roc_auc_score(y_tr, ada.predict_proba(X_tr)[:, 1]))
        depth_results.append({
            'n_estimators': n,
            'CV AUC':       np.mean(fold_val_auc),
            'Train AUC':    np.mean(fold_train_auc),
            'Gap':          np.mean(fold_train_auc) - np.mean(fold_val_auc),
        })
    results_by_depth[depth] = pd.DataFrame(depth_results).set_index('n_estimators')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {1: 'steelblue', 2: 'darkorange'}

for depth, df_n in results_by_depth.items():
    axes[0].plot(df_n.index, df_n['CV AUC'], 'o-', color=colors[depth],
                 lw=2, label=f'CV AUC (depth={depth})')
    axes[0].plot(df_n.index, df_n['Train AUC'], '--', color=colors[depth],
                 lw=1.2, alpha=0.5, label=f'Train AUC (depth={depth})')
    axes[1].plot(df_n.index, df_n['Gap'], 'o-', color=colors[depth],
                 lw=2, label=f'depth={depth}')

axes[0].set_xlabel('n_estimators'); axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('CV AUC vs n_estimators'); axes[0].legend(); axes[0].grid(True)
axes[1].set_xlabel('n_estimators'); axes[1].set_ylabel('Overfit Gap')
axes[1].set_title('Overfit Gap vs n_estimators'); axes[1].legend(); axes[1].grid(True)

plt.suptitle('AdaBoost (Decision Tree): Performance vs n_estimators', fontsize=13)
plt.tight_layout()
plt.show()

for depth, df_n in results_by_depth.items():
    best_n = df_n['CV AUC'].idxmax()
    print(f'depth={depth}: best n_estimators={best_n}  CV AUC={df_n["CV AUC"][best_n]:.4f}  Gap={df_n["Gap"][best_n]:.4f}')

In [ ]:
# Hyperparameter tuning with penalized Optuna (lambda=0.3).
# Objective: AUC - lambda * overfit_gap.
# Search space: AdaBoost-level params (n_estimators, learning_rate)
# and decision tree params (max_depth). max_depth controls tree expressiveness
# and is the primary lever for controlling bias-variance tradeoff.
# lambda=0.3: willing to give up 0.01 AUC to reduce the gap by 0.033.

lam = 0.3

def make_objective(X, y, lam):
    def objective(trial):
        n_estimators  = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 2.0, log=True)
        max_depth     = trial.suggest_int('max_depth', 1, 4)

        fold_train_auc, fold_val_auc = [], []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            base = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
            ada  = AdaBoostClassifier(
                estimator=base, n_estimators=n_estimators,
                learning_rate=learning_rate, random_state=42
            )
            ada.fit(X_tr, y_tr)
            fold_val_auc.append(roc_auc_score(y_val, ada.predict_proba(X_val)[:, 1]))
            fold_train_auc.append(roc_auc_score(y_tr, ada.predict_proba(X_tr)[:, 1]))

        val_auc = np.mean(fold_val_auc)
        gap     = np.mean(fold_train_auc) - val_auc
        return val_auc - lam * gap
    return objective

def cv_evaluate(X, y, params, skf):
    fold_train_auc, fold_val_auc, fold_logloss, fold_brier = [], [], [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        base = DecisionTreeClassifier(
            max_depth=params.get('max_depth', 1), random_state=42
        )
        ada = AdaBoostClassifier(
            estimator=base,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            random_state=42
        )
        ada.fit(X_tr, y_tr)
        proba    = ada.predict_proba(X_val)[:, 1]
        proba_tr = ada.predict_proba(X_tr)[:, 1]
        fold_val_auc.append(roc_auc_score(y_val, proba))
        fold_train_auc.append(roc_auc_score(y_tr, proba_tr))
        fold_logloss.append(log_loss(y_val, proba))
        fold_brier.append(brier_score_loss(y_val, proba))
    return {
        'CV AUC':      np.mean(fold_val_auc),
        'Train AUC':   np.mean(fold_train_auc),
        'Overfit Gap': np.mean(fold_train_auc) - np.mean(fold_val_auc),
        'Logloss':     np.mean(fold_logloss),
        'BrierScore':  np.mean(fold_brier),
    }

study_full = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_full.optimize(make_objective(X_train_clean, y_train, lam), n_trials=100, show_progress_bar=True)

best_params_full = study_full.best_params
res_full = cv_evaluate(X_train_clean, y_train, best_params_full, skf)

print(f'Best params: {best_params_full}')
print(f'CV AUC:      {res_full["CV AUC"]:.4f}')
print(f'Train AUC:   {res_full["Train AUC"]:.4f}')
print(f'Overfit Gap: {res_full["Overfit Gap"]:.4f}')
print(f'Logloss:     {res_full["Logloss"]:.4f}')
print(f'BrierScore:  {res_full["BrierScore"]:.4f}')

# Optuna optimization history and hyperparameter importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
trial_vals = [t.value for t in study_full.trials]
axes[0].plot(trial_vals, color='steelblue', alpha=0.5, linewidth=1)
axes[0].plot(np.maximum.accumulate(trial_vals), color='darkblue', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Penalized Score (AUC - 0.3 * Gap)')
axes[0].set_title('Optuna Optimization History'); axes[0].legend(); axes[0].grid(True)
importances = optuna.importance.get_param_importances(study_full)
axes[1].barh(list(importances.keys()), list(importances.values()), color='steelblue')
axes[1].set_xlabel('Importance'); axes[1].set_title('Hyperparameter Importance (Optuna)')
axes[1].grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Fit tuned model on full training set and evaluate on test set.
# Also produce ROC curve, confusion matrix, calibration curve, and PR curve.

def build_model(params):
    base = DecisionTreeClassifier(
        max_depth=params.get('max_depth', 1), random_state=42
    )
    return AdaBoostClassifier(
        estimator=base,
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        random_state=42
    )

ada_full = build_model(best_params_full)
ada_full.fit(X_train_clean, y_train)

y_proba       = ada_full.predict_proba(X_test_clean)[:, 1]
y_pred        = ada_full.predict(X_test_clean)
y_proba_train = ada_full.predict_proba(X_train_clean)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

test_auc_full  = roc_auc_score(y_test, y_proba)
train_auc_full = roc_auc_score(y_train, y_proba_train)
gap_test_full  = train_auc_full - test_auc_full

print(f'ROC-AUC (Test):  {test_auc_full:.4f}')
print(f'Log Loss:        {log_loss(y_test, y_proba):.4f}')
print(f'Brier Score:     {brier_score_loss(y_test, y_proba):.4f}')
print(f'Accuracy:        {accuracy_score(y_test, y_pred):.4f}')
print(f'F1:              {f1_score(y_test, y_pred):.4f}')
print(f'Overfit Gap:     {gap_test_full:.4f}')
print('')
print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0, 0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {test_auc_full:.4f}')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].set_xlabel('FPR'); axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curve'); axes[0, 0].legend(); axes[0, 0].grid(True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Pred: Non-hitmaker', 'Pred: Hitmaker'],
            yticklabels=['Actual: Non-hitmaker', 'Actual: Hitmaker'])
axes[0, 1].set_title('Confusion Matrix')

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
axes[1, 0].plot(prob_pred, prob_true, 's-', color='steelblue', label='AdaBoost (Tree)')
axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[1, 0].set_xlabel('Mean Predicted Probability'); axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title('Calibration Curve'); axes[1, 0].legend(); axes[1, 0].grid(True)

precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1, 1].plot(recall_vals, precision_vals, color='steelblue', lw=2, label=f'AP = {ap:.4f}')
axes[1, 1].set_xlabel('Recall'); axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('Precision-Recall Curve'); axes[1, 1].legend(); axes[1, 1].grid(True)

plt.suptitle('AdaBoost (Decision Tree, Full Features, Tuned) — Test Set Evaluation', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance on the full-feature tuned model.
# Unlike ml_sandbox_19_adaboost_linear (linear AdaBoost), decision tree AdaBoost does expose
# feature_importances_ via the ensemble, but permutation importance is more
# reliable and consistent with ml_sandbox_19_adaboost_linear for direct comparison.
# We compute on the training set to avoid using test data for feature selection.

perm_full = permutation_importance(
    ada_full, X_train_clean, y_train,
    n_repeats=20, random_state=42, scoring='roc_auc'
)

perm_df_full = pd.DataFrame({
    'Feature':    X_train_clean.columns,
    'Importance': perm_full.importances_mean,
    'Std':        perm_full.importances_std,
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Permutation importance (mean AUC drop when shuffled):')
print(perm_df_full.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 8))
plot_df = perm_df_full.sort_values('Importance', ascending=True)
colors = ['#d62728' if v >= perm_df_full['Importance'].quantile(0.75) else '#aec7e8'
          for v in plot_df['Importance']]
ax.barh(plot_df['Feature'], plot_df['Importance'], xerr=plot_df['Std'],
        color=colors, error_kw={'ecolor': 'gray', 'capsize': 3})
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Mean AUC Drop (permutation importance)')
ax.set_title('AdaBoost (Decision Tree) — Permutation Feature Importance (Full Features)')
plt.tight_layout()
plt.show()

#### Feature contribution analysis to see which features can be dropped

In [ ]:
# Ablation analysis: drop each feature one at a time and measure the impact
# on CV AUC and overfit gap. Features with near-zero or positive AUC delta
# when dropped are candidates for removal — they add noise, not signal.
# try/except handles the rare case where AdaBoost fails to converge after
# a critical feature is removed (records NaN for that feature).

def evaluate_features(X, y, params, skf):
    fold_train_auc, fold_val_auc = [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        ada = build_model(params)
        try:
            ada.fit(X_tr, y_tr)
            fold_val_auc.append(roc_auc_score(y_val, ada.predict_proba(X_val)[:, 1]))
            fold_train_auc.append(roc_auc_score(y_tr, ada.predict_proba(X_tr)[:, 1]))
        except ValueError:
            fold_val_auc.append(np.nan)
            fold_train_auc.append(np.nan)
    return np.nanmean(fold_val_auc), np.nanmean(fold_train_auc) - np.nanmean(fold_val_auc)

baseline_auc, baseline_gap = evaluate_features(X_train_clean, y_train, best_params_full, skf)
print(f'{"Baseline (all features)":50s}  AUC: {baseline_auc:.4f}  Gap: {baseline_gap:.4f}')
print('-' * 75)

ablation_results = []
for feature in X_train_clean.columns:
    X_abl = X_train_clean.drop(columns=[feature])
    auc, gap = evaluate_features(X_abl, y_train, best_params_full, skf)
    delta_auc = auc - baseline_auc
    delta_gap = gap - baseline_gap
    ablation_results.append({
        'Dropped':   feature,
        'CV AUC':    auc,
        'Delta AUC': delta_auc,
        'Gap':       gap,
        'Delta Gap': delta_gap,
    })
    print(f'Drop [{feature[:45]:45s}]  AUC: {auc:.4f} ({delta_auc:+.4f})  Gap: {gap:.4f} ({delta_gap:+.4f})')

df_ablation = pd.DataFrame(ablation_results).set_index('Dropped').sort_values('Delta AUC')

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
auc_colors = ['#2ca02c' if v >= 0 else '#d62728' for v in df_ablation['Delta AUC']]
gap_colors = ['#2ca02c' if v <= 0 else '#d62728' for v in df_ablation['Delta Gap']]
df_ablation['Delta AUC'].plot(kind='barh', ax=axes[0], color=auc_colors)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Delta CV AUC when feature dropped\n(green = dropping helps)')
df_ablation['Delta Gap'].plot(kind='barh', ax=axes[1], color=gap_colors)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Delta Overfit Gap when feature dropped\n(green = dropping helps)')
plt.suptitle('Ablation Analysis', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Genre consolidation: following ml_sandbox_17_compare_final_xgboost_tuning, ml_sandbox_18_catboost, and ml_sandbox_19_adaboost_linear,
# we consolidate 14 low-signal genre dummies into a single artist_genre_other feature,
# keeping 4 high-signal genres separate.
# This reduces feature noise while preserving genre signal as a consolidated binary flag.

high_signal_genres = [
    'artist_genre_Pop',
    'artist_genre_Rock',
    'artist_genre_Hip Hop/Rap',
    'artist_genre_R&B/Soul/Funk',
]
all_genre_cols    = [c for c in X_train_clean.columns if c.startswith('artist_genre_')]
low_signal_genres = [c for c in all_genre_cols if c not in high_signal_genres]

X_train_cons = X_train_clean.drop(columns=low_signal_genres).copy()
X_train_cons['artist_genre_other'] = (X_train_clean[low_signal_genres].sum(axis=1) > 0).astype(int)
X_test_cons  = X_test_clean.drop(columns=low_signal_genres).copy()
X_test_cons['artist_genre_other']  = (X_test_clean[low_signal_genres].sum(axis=1) > 0).astype(int)

print(f'Features before consolidation: {X_train_clean.shape[1]}')
print(f'Features after consolidation:  {X_train_cons.shape[1]}')
print('')
print('Consolidated feature list:')
print(X_train_cons.columns.tolist())

# Permutation importance on consolidated set — used to guide forward selection order
ada_cons = build_model(best_params_full)
ada_cons.fit(X_train_cons, y_train)

perm_cons = permutation_importance(
    ada_cons, X_train_cons, y_train,
    n_repeats=20, random_state=42, scoring='roc_auc'
)
feat_imp_cons = pd.DataFrame({
    'Feature':    X_train_cons.columns,
    'Importance': perm_cons.importances_mean,
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feature_order = feat_imp_cons['Feature'].tolist()

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp_plot = feat_imp_cons.sort_values('Importance', ascending=True)
colors = ['#d62728' if v >= feat_imp_cons['Importance'].quantile(0.75) else '#aec7e8'
          for v in feat_imp_plot['Importance']]
ax.barh(feat_imp_plot['Feature'], feat_imp_plot['Importance'], color=colors)
ax.axvline(feat_imp_cons['Importance'].mean(), color='black', linestyle='--',
           linewidth=1, label='Mean')
ax.set_xlabel('Mean AUC Drop (permutation importance)')
ax.set_title('AdaBoost (Decision Tree) — Permutation Feature Importance (Consolidated Set)')
ax.legend()
plt.tight_layout()
plt.show()

print('')
print('Feature importance order (for forward selection):')
print(feat_imp_cons.to_string(index=False))

### Re-tuning AdaBoost (Decision Tree) with genre consolidation and forward feature selection

In [ ]:
# Forward feature selection: start with top 3 features (by permutation importance)
# and add one at a time. Track CV AUC, overfit gap, Logloss, and BrierScore
# to find the optimal feature count.

sel_results = []
for n_feats in range(3, len(feature_order) + 1):
    feats = feature_order[:n_feats]
    X_sub = X_train_cons[feats]
    fold_train_auc, fold_val_auc, fold_logloss, fold_brier = [], [], [], []
    for train_idx, val_idx in skf.split(X_sub, y_train):
        X_tr, X_val = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        ada = build_model(best_params_full)
        try:
            ada.fit(X_tr, y_tr)
            proba    = ada.predict_proba(X_val)[:, 1]
            proba_tr = ada.predict_proba(X_tr)[:, 1]
            fold_val_auc.append(roc_auc_score(y_val, proba))
            fold_train_auc.append(roc_auc_score(y_tr, proba_tr))
            fold_logloss.append(log_loss(y_val, proba))
            fold_brier.append(brier_score_loss(y_val, proba))
        except ValueError:
            fold_val_auc.append(np.nan); fold_train_auc.append(np.nan)
            fold_logloss.append(np.nan); fold_brier.append(np.nan)
    val_auc   = np.nanmean(fold_val_auc)
    train_auc = np.nanmean(fold_train_auc)
    sel_results.append({
        'n_features':  n_feats,
        'last_added':  feature_order[n_feats - 1],
        'CV AUC':      val_auc,
        'Train AUC':   train_auc,
        'Overfit Gap': train_auc - val_auc,
        'Logloss':     np.nanmean(fold_logloss),
        'BrierScore':  np.nanmean(fold_brier),
    })
    print(f'n={n_feats:2d}  +[{feature_order[n_feats-1][:40]:40s}]  AUC: {val_auc:.4f}  Gap: {train_auc-val_auc:.4f}')

df_sel = pd.DataFrame(sel_results).set_index('n_features')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (metric, color, better) in zip(axes.flat, [
    ('CV AUC',      '#1f77b4', 'higher'),
    ('Overfit Gap', '#d62728', 'lower'),
    ('Logloss',     '#ff7f0e', 'lower'),
    ('BrierScore',  '#2ca02c', 'lower'),
]):
    ax.plot(df_sel.index, df_sel[metric], 'o-', color=color, linewidth=2, markersize=5)
    best_n   = df_sel[metric].idxmax() if better == 'higher' else df_sel[metric].idxmin()
    best_val = df_sel[metric][best_n]
    ax.scatter([best_n], [best_val], color='black', zorder=5, s=80)
    ax.axvline(best_n, color='black', linestyle='--', linewidth=0.8)
    ax.text(best_n + 0.15, best_val, f'n={best_n}', fontsize=9, va='center')
    ax.set_xlabel('Number of Features'); ax.set_ylabel(metric)
    ax.set_title(f'{metric}  ({"higher" if better == "higher" else "lower"} is better)')
    ax.grid(True)
plt.suptitle('Forward Selection — Consolidated Feature Set', fontsize=13)
plt.tight_layout()
plt.show()

print('')
print(df_sel[['CV AUC', 'Overfit Gap', 'Logloss', 'BrierScore']].round(4).to_string())

In [ ]:
# Re-tune with penalized Optuna (lambda=0.3) on the optimal consolidated feature subset.
# Pick n_optimal based on the forward selection plot above (best AUC / acceptable gap).
# Re-tuning on the reduced feature set allows hyperparameters to adapt to the new feature space.

n_optimal     = df_sel['CV AUC'].idxmax()   # adjust manually if desired
top_features  = feature_order[:n_optimal]
X_train_final = X_train_cons[top_features]
X_test_final  = X_test_cons[top_features]

print(f'Final feature set ({n_optimal} features):')
for i, f in enumerate(top_features, 1):
    print(f'  {i:2d}. {f}')

study_final = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_final.optimize(make_objective(X_train_final, y_train, lam), n_trials=100, show_progress_bar=True)

best_params_final = study_final.best_params
res_final = cv_evaluate(X_train_final, y_train, best_params_final, skf)

print('')
print(f'Best params: {best_params_final}')
print(f'CV AUC:      {res_final["CV AUC"]:.4f}')
print(f'Train AUC:   {res_final["Train AUC"]:.4f}')
print(f'Overfit Gap: {res_final["Overfit Gap"]:.4f}')
print(f'Logloss:     {res_final["Logloss"]:.4f}')
print(f'BrierScore:  {res_final["BrierScore"]:.4f}')

In [ ]:
# Compare all AdaBoost (Decision Tree) candidates on the held-out test set.
# AdaBoost (linear, ml_sandbox_19_adaboost_linear), CatBoost (ml_sandbox_18_catboost), and XGBoost (ml_sandbox_17_compare)
# results are included as references.
# All models are fit on the full training set and evaluated once on the test set.

def test_evaluate(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    proba_te = model.predict_proba(X_te)[:, 1]
    proba_tr = model.predict_proba(X_tr)[:, 1]
    return {
        'Test AUC':    roc_auc_score(y_te, proba_te),
        'Train AUC':   roc_auc_score(y_tr, proba_tr),
        'Overfit Gap': roc_auc_score(y_tr, proba_tr) - roc_auc_score(y_te, proba_te),
        'Logloss':     log_loss(y_te, proba_te),
        'BrierScore':  brier_score_loss(y_te, proba_te),
    }

candidates = {
    'AdaBoost Tree (all 26 features)': (
        build_model(best_params_full),
        X_train_clean, X_test_clean
    ),
    f'AdaBoost Tree (consolidated n={n_optimal})': (
        build_model(best_params_final),
        X_train_final, X_test_final
    ),
}

test_results = []
for name, (model, X_tr, X_te) in candidates.items():
    res = test_evaluate(model, X_tr, y_train, X_te, y_test)
    res['Model']      = name
    res['N Features'] = X_tr.shape[1]
    test_results.append(res)
    print(f'{name}: AUC={res["Test AUC"]:.4f}  Gap={res["Overfit Gap"]:.4f}  Logloss={res["Logloss"]:.4f}  Brier={res["BrierScore"]:.4f}')

# Reference results from prior notebooks
test_results.append({
    'Model': 'AdaBoost Linear (ml_sandbox_19_adaboost_linear)', 'Test AUC': 0.7436,
    'Train AUC': 0.7415, 'Overfit Gap': -0.0021,
    'Logloss': 0.5856, 'BrierScore': 0.1986, 'N Features': 11,
})
test_results.append({
    'Model': 'CatBoost (ml_sandbox_18_catboost, consolidated n=10)', 'Test AUC': 0.7634,
    'Train AUC': 0.7888, 'Overfit Gap': 0.0254,
    'Logloss': None, 'BrierScore': None, 'N Features': 10,
})
test_results.append({
    'Model': 'XGBoost (ml_sandbox_17_compare)', 'Test AUC': 0.773,
    'Train AUC': 0.806, 'Overfit Gap': 0.047,
    'Logloss': 0.558, 'BrierScore': 0.189, 'N Features': 22,
})

df_comparison = pd.DataFrame(test_results).set_index('Model').round(4)
print('')
print('--- Final Comparison ---')
print(df_comparison[['Test AUC', 'Overfit Gap', 'Logloss', 'BrierScore', 'N Features']].to_string())

## Final model analysis

In [ ]:
# Final model: AdaBoost (Decision Tree) consolidated n=optimal, fitted on the full training set.
# Evaluation on the held-out test set with full metrics, plots, and permutation importance.
# Feature direction table inferred from mean feature value for hitmakers vs non-hitmakers.

ada_model_final = build_model(best_params_final)
ada_model_final.fit(X_train_final, y_train)

y_proba       = ada_model_final.predict_proba(X_test_final)[:, 1]
y_pred        = ada_model_final.predict(X_test_final)
y_proba_train = ada_model_final.predict_proba(X_train_final)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 60)
print(f'  FINAL MODEL -- AdaBoost Tree (consolidated n={n_optimal})')
print('=' * 60)
print(f'  Features:       {len(top_features)}')
print(f'  Test samples:   {len(y_test)}  (train: {len(y_train)})')
print('')
print('  -- Test --')
print(f'  ROC-AUC:        {roc_auc_score(y_test, y_proba):.4f}')
print(f'  Avg Precision:  {average_precision_score(y_test, y_proba):.4f}')
print(f'  Log Loss:       {log_loss(y_test, y_proba):.4f}')
print(f'  Brier Score:    {brier_score_loss(y_test, y_proba):.4f}')
print(f'  Accuracy:       {accuracy_score(y_test, y_pred):.4f}')
print(f'  F1:             {f1_score(y_test, y_pred):.4f}')
print(f'  Precision:      {precision_score(y_test, y_pred):.4f}')
print(f'  Recall:         {recall_score(y_test, y_pred):.4f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print('')
print('  -- Train --')
print(f'  ROC-AUC:        {roc_auc_score(y_train, y_proba_train):.4f}')
print(f'  Log Loss:       {log_loss(y_train, y_proba_train):.4f}')
print('')
print('  -- Overfit Gap --')
print(f'  Train AUC - Test AUC: {roc_auc_score(y_train, y_proba_train) - roc_auc_score(y_test, y_proba):.4f}')
print('')
print(classification_report(y_test, y_pred, target_names=['One-hit Wonder', 'Hitmaker']))

# Figure 1: ROC, Confusion Matrix, Calibration, PR curve
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0, 0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc_score(y_test, y_proba):.4f}')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].set_xlabel('FPR'); axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curve'); axes[0, 0].legend(); axes[0, 0].grid(True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Pred: Non-hitmaker', 'Pred: Hitmaker'],
            yticklabels=['Actual: Non-hitmaker', 'Actual: Hitmaker'])
axes[0, 1].set_title('Confusion Matrix')

fraction_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
axes[1, 0].plot(mean_pred, fraction_pos, 's-', color='steelblue', label='AdaBoost (Tree)')
axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[1, 0].set_xlabel('Mean Predicted Probability'); axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title('Calibration Curve'); axes[1, 0].legend(); axes[1, 0].grid(True)

precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1, 1].plot(recall_vals, precision_vals, color='steelblue', lw=2, label=f'AP = {ap:.4f}')
axes[1, 1].set_xlabel('Recall'); axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('Precision-Recall Curve'); axes[1, 1].legend(); axes[1, 1].grid(True)

plt.suptitle(f'Final AdaBoost (Decision Tree) Model — Test Set Evaluation', fontsize=13)
plt.tight_layout()
plt.show()

# Figure 2: Permutation importance
perm_final = permutation_importance(
    ada_model_final, X_train_final, y_train,
    n_repeats=20, random_state=42, scoring='roc_auc'
)
perm_final_df = pd.DataFrame({
    'Feature':    X_train_final.columns,
    'Importance': perm_final.importances_mean,
    'Std':        perm_final.importances_std,
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#d62728' if v >= perm_final_df['Importance'].quantile(0.75) else '#aec7e8'
          for v in perm_final_df['Importance']]
ax.barh(perm_final_df['Feature'], perm_final_df['Importance'],
        xerr=perm_final_df['Std'], color=colors,
        error_kw={'ecolor': 'gray', 'capsize': 3})
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Mean AUC Drop (permutation importance)')
ax.set_title('Final AdaBoost (Decision Tree) — Permutation Feature Importance')
plt.tight_layout()
plt.show()

# Feature direction table
hitmaker_mask = y_train == 1
direction_df = pd.DataFrame({
    'Perm Importance':      perm_final_df.set_index('Feature')['Importance'],
    'Mean (hitmakers)':     X_train_final[hitmaker_mask].mean(),
    'Mean (non-hitmakers)': X_train_final[~hitmaker_mask].mean(),
}).sort_values('Perm Importance', ascending=False).round(4)
direction_df['Direction'] = (direction_df['Mean (hitmakers)'] > direction_df['Mean (non-hitmakers)']).map(
    {True: 'Higher → Hitmaker', False: 'Lower → Hitmaker'}
)
print('')
print('Feature direction (based on mean feature value for hitmakers vs non-hitmakers):')
print(direction_df.to_string())

# Threshold tuning
thresholds = np.arange(0.1, 0.9, 0.01)
precisions, recalls, f1s = [], [], []
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

precisions = np.array(precisions)
recalls    = np.array(recalls)
f1s        = np.array(f1s)
best_f1_thresh = thresholds[np.argmax(f1s)]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, precisions, label='Precision (Hitmaker)', color='steelblue',  lw=2)
ax.plot(thresholds, recalls,    label='Recall (Hitmaker)',    color='darkorange', lw=2)
ax.plot(thresholds, f1s,        label='F1 (Hitmaker)',        color='green',      lw=2)
ax.axvline(0.5,            color='k',     linestyle='--', lw=1.2, label='Default (0.5)')
ax.axvline(best_f1_thresh, color='green', linestyle=':',  lw=1.5, label=f'Best F1 ({best_f1_thresh:.2f})')
ax.set_xlabel('Classification Threshold', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Threshold Tuning — AdaBoost (Decision Tree) Final Model', fontsize=15)
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n{"Threshold":>10} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 42)
for t in [0.35, 0.40, 0.45, 0.50, best_f1_thresh]:
    y_pred_t = (y_proba >= t).astype(int)
    print(f'{t:>10.2f} {precision_score(y_test, y_pred_t):>10.3f} '
          f'{recall_score(y_test, y_pred_t):>8.3f} '
          f'{f1_score(y_test, y_pred_t):>8.3f}')